# VerdaSense — RAG Ingestion Pipeline (v2) 

In [2]:
#  ╔══════════════════════════════════════════════════════════════════════════════╗
#  ║  wound_dressing_rag_ingestion_v2.ipynb                                       ║
#  ║                                                                              ║
#  ║                                                                              ║
#  ║  WHAT CHANGED FROM v1:                                                       ║
#  ║  • Reads from curated chunk JSONs (chunk_visualiser exports) instead of      ║
#  ║    re-running partition_pdf — skipping is preserved from  manual review      ║
#  ║  • Improved wound-care-specific AI summary prompt for tables/images          ║
#  ║  • Tighter T.I.M.E. tag extraction (clinical phrases, not single words)      ║
#  ║  • Per-source guideline metadata (authority, year, guideline_type)           ║
#  ║  • has_table / has_image flags carried through for retrieval filtering       ║
#  ║  • Non-English / non-clinical chunks skipped automatically                   ║
#  ║  • DB and PDF dirs updated to v2                                             ║
#  ╚══════════════════════════════════════════════════════════════════════════════╝

## 1. Imports

In [10]:
import os
import json
import time
import random
import torch
import tiktoken
import shutil
from typing import List, Optional, Dict, Any

from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title

from langchain_core.documents import Document
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv

load_dotenv()
print("Imports OK")
print(f"GPU available: {torch.cuda.is_available()}")

Imports OK
GPU available: True


## 2. Path Configs

In [16]:
PDF_DIR = "./clinical_pdfs_v2/"
DB_DIR = "./db_wound_care_v2"
CHUNKS_DIR = "./chunks_selection/"

# ── Curated chunk JSON exports from chunk_visualiser ─────────────────────────
CHUNK_JSON_FILES = {
    "AJGP_kept.json"          : os.path.join(CHUNKS_DIR, "AJGP_kept.json"),
    "GarisPanduan_kept.json"  : os.path.join(CHUNKS_DIR, "GarisPanduan_kept.json"),
    "SFP_kept.json"           : os.path.join(CHUNKS_DIR, "SFP_kept.json"),
    "WoundCare_kept.json"     : os.path.join(CHUNKS_DIR, "WoundCare_kept.json"),
}

# ── Per-source guideline metadata ─────────────────────────────────────────────
GUIDELINE_METADATA = {
    "AJGP-11-2022-Focus-Sinha-Wound-Dressings-WEB.pdf": {
        "guideline_type" : "clinical_review",
        "authority"      : "RACGP_Australia",
        "year"           : "2022",
        "focus"          : "acute_wound_dressings_general_practice",
    },
    "2_garis_panduan_perkhidmatan_penjagaan_luka_di_fasiliti_kesihatan_primer.pdf": {
        "guideline_type" : "national_guideline",
        "authority"      : "MOH_Malaysia",
        "year"           : "2019",
        "focus"          : "primary_care_wound_service",
    },
    "SFP-Vol403.Unit2.pdf": {
        "guideline_type" : "clinical_education",
        "authority"      : "Singapore_Family_Physician",
        "year"           : "2018",
        "focus"          : "wound_management_primary_care",
    },
    "Wound_Care_Manual.pdf": {
        "guideline_type" : "clinical_manual",
        "authority"      : "MOH_Malaysia",
        "year"           : "2014",
        "focus"          : "comprehensive_wound_care",
    },
    # Legacy — keep so old chunks still resolve
    # "Book2-wound-dressing-guide.pdf": {
    #     "guideline_type" : "dressing_properties",
    #     "authority"      : "QUT_Australia",
    #     "year"           : "2013",
    #     "focus"          : "dressing_categories_properties",
    # },
    # "HDFT-Wound-Dressing-Guideline-2018-v2.1-PDF.pdf": {
    #     "guideline_type" : "hospital_protocol",
    #     "authority"      : "HDFT_UK",
    #     "year"           : "2018",
    #     "focus"          : "wound_dressing_selection",
    # },
}

# ── Verify paths exist ────────────────────────────────────────────────────────
print(f"\n📂 Path check:")
print(f"   PDF_DIR    : {PDF_DIR}   {'✅' if os.path.isdir(PDF_DIR) else '⚠️  not found (reference only)'}")
print(f"   DB_DIR     : {DB_DIR}   {'✅ exists (will overwrite)' if os.path.isdir(DB_DIR) else '(will be created)'}")
print(f"   CHUNKS_DIR : {CHUNKS_DIR}  {'✅' if os.path.isdir(CHUNKS_DIR) else '❌ NOT FOUND'}")
 
print(f"\n📋 Chunk JSON files:")
all_json_ok = True
for name, path in CHUNK_JSON_FILES.items():
    exists = os.path.isfile(path)
    print(f"   {'✅' if exists else '❌'} {name}")
    if not exists:
        all_json_ok = False
 
if not all_json_ok:
    print("\n⚠️  One or more JSON files not found. Update CHUNK_JSON_FILES paths above.")
else:
    print("\n✅ All JSON files found.")
 
 


📂 Path check:
   PDF_DIR    : ./clinical_pdfs_v2/   ✅
   DB_DIR     : ./db_wound_care_v2   (will be created)
   CHUNKS_DIR : ./chunks_selection/  ✅

📋 Chunk JSON files:
   ✅ AJGP_kept.json
   ✅ GarisPanduan_kept.json
   ✅ SFP_kept.json
   ✅ WoundCare_kept.json

✅ All JSON files found.


## 3. Rate limit helpers

In [17]:
def count_tokens(text: str, model: str = "gpt-4o-mini") -> int:
    """Estimate tokens in a string using tiktoken before sending to OpenAI."""
    try:
        enc = tiktoken.encoding_for_model(model)
    except KeyError:
        enc = tiktoken.get_encoding("cl100k_base")
    return len(enc.encode(text))
 
 
def call_with_backoff(fn, max_retries: int = 6, base_delay: float = 1.0):
    """
    Calls fn() and retries on 429 rate-limit errors using exponential backoff.
    Parses OpenAI's suggested retry delay when available.
    """
    for attempt in range(max_retries):
        try:
            return fn()
        except Exception as e:
            err_str = str(e)
            is_rate_limit = "429" in err_str or "rate_limit_exceeded" in err_str
 
            if is_rate_limit and attempt < max_retries - 1:
                retry_ms = None
                if "Please try again in" in err_str:
                    try:
                        after_part = err_str.split("Please try again in")[1]
                        ms_str = after_part.strip().split("ms")[0].strip()
                        retry_ms = float(ms_str) / 1000.0
                    except Exception:
                        pass
 
                if retry_ms:
                    wait = retry_ms + 0.5
                    print(f"     ⏳ Rate limit — OpenAI says wait {retry_ms:.1f}s, sleeping {wait:.1f}s...")
                else:
                    wait = base_delay * (2 ** attempt) * (0.8 + random.random() * 0.4)
                    print(f"     ⏳ Rate limit (attempt {attempt+1}/{max_retries}) — sleeping {wait:.1f}s...")
 
                time.sleep(wait)
            else:
                raise
 
    raise RuntimeError(f"Max retries ({max_retries}) exceeded")
 
 
class TPMThrottle:
    """
    Proactively tracks tokens-per-minute and sleeps before hitting the limit.
    Much better than reacting to 429s after they happen.
    """
    def __init__(self, tpm_limit: int = 200_000, safety_margin: float = 0.85):
        self.tpm_limit     = tpm_limit
        self.safety_margin = safety_margin
        self.window_start  = time.time()
        self.tokens_used   = 0
 
    def _reset_if_new_window(self):
        if time.time() - self.window_start >= 60.0:
            self.window_start = time.time()
            self.tokens_used  = 0
 
    def check_and_wait(self, tokens_needed: int):
        """Sleep until next window if this request would exceed the safe ceiling."""
        self._reset_if_new_window()
        soft_limit = self.tpm_limit * self.safety_margin
        if self.tokens_used + tokens_needed > soft_limit:
            elapsed   = time.time() - self.window_start
            remaining = 60.0 - elapsed + 1.0
            if remaining > 0:
                print(f"     🕐 TPM ceiling reached ({self.tokens_used:,} / {int(soft_limit):,} tokens). "
                      f"Waiting {remaining:.1f}s for new window...")
                time.sleep(remaining)
                self.window_start = time.time()
                self.tokens_used  = 0
 
    def record(self, tokens_used: int):
        """Record tokens consumed after a successful API call."""
        self._reset_if_new_window()
        self.tokens_used += tokens_used
 
 
# One shared throttle for the entire pipeline run
throttle = TPMThrottle(tpm_limit=200_000, safety_margin=0.85)
print("✅ Rate limit helpers ready")

✅ Rate limit helpers ready


## 4. Curated chunk loader (NEW - Replaces Partition PDF)

In [18]:
def load_curated_chunks(json_paths: List[str]) -> List[dict]:
    """
    Load manually curated chunk JSONs exported from chunk_visualiser.
    Each JSON has the structure:
        {
          "meta": { ... },
          "kept_chunks": [
              {
                "chunk_id": int,
                "source": str,          ← PDF filename
                "pages": [int, ...],
                "char_count": int,
                "time_tags": [str, ...],
                "has_table": bool,
                "has_image": bool,
                "is_english": bool,
                "text": str
              },
              ...
          ]
        }
 
    Returns a flat list of chunk dicts across all JSON files.
    """
    all_chunks = []
    for path in json_paths:
        if not os.path.isfile(path):
            print(f"   ⚠️  Skipping missing file: {path}")
            continue
 
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
 
        kept = data.get("kept_chunks", [])
        meta = data.get("meta", {})
        print(f"   📄 {os.path.basename(path)}: "
              f"{len(kept)} kept chunks "
              f"(of {meta.get('total_chunks', '?')} total, "
              f"{meta.get('skipped_count', '?')} skipped)")
        all_chunks.extend(kept)
 
    print(f"\n   Total curated chunks loaded: {len(all_chunks)}")
    return all_chunks

## 5. Improved T.I.M.E Tags Extraction

In [19]:
TIME_KEYWORDS_V2 = {
    "T": [
        "necrotic tissue", "slough", "granulat", "eschar", "fibrinous",
        "necrosis", "tissue viability", "black tissue", "yellow tissue",
        "wound bed tissue", "tissue type", "devitalised", "devitalized",
        "non-viable tissue", "viable tissue",
    ],
    "I": [
        "infected wound", "wound infection", "biofilm", "antimicrobial dressing",
        "silver dressing", "cadexomer", "erythema", "cellulitis", "purulent",
        "critically colonised", "critically colonized", "bacterial burden",
        "sepsis", "iodine dressing", "signs of infection", "wound colonis",
        "antimicrobial agent", "systemic antibiotic",
    ],
    "M": [
        "exudate", "maceration", "desicat", "heavily exud", "copious drainage",
        "moisture balance", "exuding wound", "wet wound", "dry wound bed",
        "absorbent dressing", "hydrofiber", "alginate", "moisture imbalance",
        "wound drainage", "exudate level", "periwound maceration",
        "moisture retentive", "moist wound",
    ],
    "E": [
        "wound edge", "epidermal margin", "epithelial migration", "wound margin",
        "periwound skin", "undermining", "rolled edge", "epibole",
        "advancing edge", "non-advancing", "epithelialis", "wound border",
        "wound contraction", "wound closure", "edge advancement",
    ],
}
 
def extract_time_tags_v2(text: str) -> str:
    """
    Scan chunk text for T.I.M.E. clinical phrases.
    Returns comma-separated string (e.g. "T,M") or "none".
    ChromaDB does NOT accept empty lists — always returns a string.
    """
    tags = []
    t_lower = text.lower()
    for tag, keywords in TIME_KEYWORDS_V2.items():
        if any(kw in t_lower for kw in keywords):
            tags.append(tag)
    return ",".join(tags) if tags else "none"
 
 
# Quick self-test
_test = "The wound shows necrotic tissue with heavy exudate and signs of infection at the wound edge."
_expected = {"T", "I", "M", "E"}
_got = set(extract_time_tags_v2(_test).split(","))
assert _got == _expected, f"Tag extraction self-test failed: got {_got}"
print(f"✅ T.I.M.E. tag extraction ready (self-test passed: {_got})")

✅ T.I.M.E. tag extraction ready (self-test passed: {'M', 'E', 'T', 'I'})


## 6. Improved AI summary prompt

In [20]:
def create_ai_enhanced_summary(
    text: str,
    tables: List[str],
    images: List[str],
    source: str = "",
) -> Optional[str]:
    """
    Send text + tables + images to GPT-4o-mini for a structured clinical extraction.
 
    v2 improvements over v1:
    - Wound-care-specific extraction prompt (not generic summarisation)
    - Explicitly asks for: dressing triggers, contraindications, change frequency,
      referral criteria, T.I.M.E. framework references
    - Returns None for non-clinical chunks (flagged with SKIP) so they are dropped
    - Falls back to raw text if all retries exhausted
 
    Uses TPMThrottle (proactive) + call_with_backoff (reactive).
    """
    try:
        llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
 
        # ── Build prompt ──────────────────────────────────────────────────────
        prompt_text = (
            "You are a clinical wound care expert extracting structured knowledge\n"
            "from clinical guideline documents for a wound dressing recommendation system.\n\n"
            f"SOURCE DOCUMENT: {source}\n\n"
            "DOCUMENT CONTENT:\n"
            f"{text}\n\n"
        )
        if tables:
            prompt_text += "TABLES (HTML format — extract all dressing data from these):\n"
            prompt_text += "\n".join(tables) + "\n\n"
 
        prompt_text += (
            "EXTRACTION TASK:\n"
            "Extract and summarise the clinical wound care knowledge in this content.\n"
            "Focus specifically on:\n"
            "1. Which wound characteristics (tissue type, infection status, exudate level,\n"
            "   wound edge) trigger which dressing recommendation\n"
            "2. Specific dressing product names, categories, and brand examples mentioned\n"
            "3. Contraindications — what NOT to use and why\n"
            "4. Dressing change frequency and duration of use\n"
            "5. When to refer the patient or escalate care\n"
            "6. Any T.I.M.E. framework references (Tissue, Infection, Moisture, Edge)\n"
            "7. Debridement requirements if mentioned\n\n"
            "Write in clear clinical English only. Use bullet points for lists.\n"
            "If the content is NOT about wound care, dressings, or wound management,\n"
            "write exactly: SKIP\n\n"
            "EXTRACTED CLINICAL KNOWLEDGE:"
        )
 
        message_content = [{"type": "text", "text": prompt_text}]
        for image_b64 in images:
            message_content.append({
                "type": "image_url",
                "image_url": {"url": f"data:image/jpeg;base64,{image_b64}"},
            })
 
        # Proactive throttle — estimate tokens before sending
        tokens_needed = count_tokens(prompt_text) + 500   # +500 for response
        throttle.check_and_wait(tokens_needed)
 
        def api_call():
            return llm.invoke([HumanMessage(content=message_content)])
 
        response = call_with_backoff(api_call)
        throttle.record(tokens_needed)
 
        result = response.content.strip()
 
        # Honour the LLM's SKIP signal — drop non-clinical chunks
        if result.upper().startswith("SKIP"):
            return None
 
        return result
 
    except Exception as e:
        print(f"     ❌ AI summary failed after retries: {e}")
        # Graceful fallback — raw text so the chunk is still indexed
        fallback = text[:600]
        if tables:
            fallback += f"\n[Contains {len(tables)} table(s) with dressing recommendation data]"
        if images:
            fallback += f"\n[Contains {len(images)} clinical image(s)]"
        return fallback
 
 
print("✅ AI summary function ready")

✅ AI summary function ready


## 7. Chunk → LangChain Document converter

In [21]:
def convert_curated_chunks_to_documents(
    curated_chunks: List[dict],
    guideline_metadata: dict,
) -> List[Document]:
    """
    Convert curated chunk dicts (from chunk_visualiser JSON exports) into
    LangChain Document objects ready for embedding.
 
    For each chunk:
    1. If it has table or image content → run AI summary (wound-care-specific prompt)
    2. If text-only → use raw text directly (no API call needed)
    3. Skip chunks flagged as SKIP by the LLM (non-clinical content)
    4. Attach full metadata: source, chunk_id, time_tags, guideline_type, etc.
 
    The chunk_visualiser JSON includes has_table / has_image flags but does NOT
    preserve the raw HTML table content or base64 images — those were in the
    unstructured output which we don't re-run. For v2 we therefore:
    - Send the text to the AI summary when has_table=True (AI can still extract
      structured info from the text representation of the table)
    - Skip image-only content gracefully
    """
    documents     = []
    total         = len(curated_chunks)
    skipped_llm   = 0
    ai_summarised = 0
    raw_text      = 0
 
    print(f"🧠 Converting {total} curated chunks to Documents...")
 
    for i, chunk in enumerate(curated_chunks, start=1):
        text      = chunk.get("text", "").strip()
        source    = chunk.get("source", "unknown.pdf")
        chunk_id  = chunk.get("chunk_id", i)
        has_table = chunk.get("has_table", False)
        has_image = chunk.get("has_image", False)
        pages     = chunk.get("pages", [])
        char_count = chunk.get("char_count", len(text))
 
        # Progress print every 5 chunks
        if i % 5 == 0 or i == 1 or i == total:
            print(f"   [{i:>3}/{total}] {source[:40]} — chunk #{chunk_id} "
                  f"({'table' if has_table else ''}"
                  f"{'img' if has_image else ''}"
                  f"{'text' if not has_table and not has_image else ''})"
                  f" {char_count} chars")
 
        if not text:
            print(f"     ⚠️  Empty text, skipping chunk #{chunk_id}")
            continue
 
        # ── Decide: AI summary or raw text ────────────────────────────────────
        if has_table or has_image:
            # has_table=True: chunk contains a dressing table — worth AI extraction
            # Note: chunk_visualiser JSON does not carry raw table HTML,
            # so we pass the text representation (unstructured flattens tables to text)
            # AI can still extract structure from this
            print(f"     → AI extraction (has_table={has_table}, has_image={has_image})...")
 
            # We pass empty lists for tables/images since they aren't in the JSON
            # The text already contains the table content as flattened text
            page_content = create_ai_enhanced_summary(
                text   = text,
                tables = [],   # table HTML not in JSON — text repr is in `text`
                images = [],   # image b64 not in JSON
                source = source,
            )
 
            if page_content is None:
                print(f"     → LLM flagged as SKIP (non-clinical), dropping chunk #{chunk_id}")
                skipped_llm += 1
                continue
 
            print(f"     → Preview: {page_content[:120]}...")
            ai_summarised += 1
 
        else:
            # Pure text chunk — use directly, no API call
            page_content = text
            raw_text += 1
 
        # ── Build metadata ─────────────────────────────────────────────────────
        # Look up guideline metadata by PDF filename
        guide_meta = guideline_metadata.get(source, {})
 
        # Re-extract T.I.M.E. tags from final page_content
        # (may differ from chunk_visualiser's tags if AI summary rewrote content)
        time_tags = extract_time_tags_v2(page_content)
 
        # Also carry the visualiser's tags as a cross-reference
        visualiser_tags = ",".join(chunk.get("time_tags", [])) or "none"
 
        metadata = {
            # ── Core identification ─────────────────────────────────
            "source"          : source,
            "chunk_id"        : chunk_id,
            "chunk_index"     : chunk_id,   # alias for backward compat with retrieval
            # ── Content flags ───────────────────────────────────────
            "has_table"       : str(has_table),   # ChromaDB stores as string
            "has_image"       : str(has_image),
            # ── T.I.M.E. tagging ───────────────────────────────────
            "time_tags"       : time_tags,
            "time_tags_raw"   : visualiser_tags,  # from visualiser (pre-AI)
            # ── Page location ───────────────────────────────────────
            "pages"           : json.dumps(pages),  # stored as JSON string
            "primary_page"    : str(pages[0]) if pages else "0",
            # ── Guideline provenance ────────────────────────────────
            "guideline_type"  : guide_meta.get("guideline_type",  "unknown"),
            "authority"       : guide_meta.get("authority",       "unknown"),
            "year"            : guide_meta.get("year",            "unknown"),
            "focus"           : guide_meta.get("focus",           "unknown"),
            # ── Processing info ─────────────────────────────────────
            "processing"      : "ai_summary" if (has_table or has_image) else "raw_text",
            "original_content": json.dumps({
                "raw_text"    : text,
                "tables_html" : [],   # not available from visualiser JSON
                "images_base64": [],  # not available from visualiser JSON
            }),
        }
 
        documents.append(Document(page_content=page_content, metadata=metadata))
 
    # ── Summary ───────────────────────────────────────────────────────────────
    print(f"\n✅ Conversion complete:")
    print(f"   Documents created : {len(documents)}")
    print(f"   AI-summarised     : {ai_summarised}  (chunks with tables/images)")
    print(f"   Raw text          : {raw_text}        (text-only chunks)")
    print(f"   Skipped (LLM)     : {skipped_llm}      (non-clinical flagged by LLM)")
 
    return documents
 
 
print("✅ Chunk → Document converter ready")
 

✅ Chunk → Document converter ready


## 8. Vector Store Creation

In [22]:
def create_vector_store(documents: List[Document], persist_directory: str):
    """
    Embed all documents with MedEmbed-large-v0.1 and persist to ChromaDB.
    Uses cosine similarity (HNSW index).
    Must use the same embedding model as the retrieval pipeline.
    """
    print(f"🔮 Creating embeddings with MedEmbed-large-v0.1...")
    print(f"   Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
 
    embedding_model = HuggingFaceEmbeddings(
        model_name="abhinand/MedEmbed-large-v0.1",
        model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"},
        encode_kwargs={"normalize_embeddings": True},
    )
 
    print(f"   Persisting to: {persist_directory}")
    print(f"   Indexing {len(documents)} documents...")
 
    vectorstore = Chroma.from_documents(
        documents=documents,
        embedding=embedding_model,
        persist_directory=persist_directory,
        collection_metadata={"hnsw:space": "cosine"},
    )
 
    print(f"✅ Vector store saved → {persist_directory}")
    return vectorstore
 
 
print("✅ Vector store builder ready")

✅ Vector store builder ready


## 9. Ingestion pipeline diagnostics helper

In [23]:
def print_pipeline_summary(documents: List[Document]):
    """Print a breakdown of the indexed documents for verification."""
    from collections import Counter
 
    print("\n" + "═" * 60)
    print("  INGESTION SUMMARY")
    print("═" * 60)
 
    # By source
    sources = Counter(d.metadata["source"] for d in documents)
    print(f"\n📚 Documents by source ({len(sources)} PDFs):")
    for src, count in sorted(sources.items()):
        authority = None
        for doc in documents:
            if doc.metadata["source"] == src:
                authority = doc.metadata.get("authority", "unknown")
                break
        print(f"   {count:>3}  {src[:55]}  [{authority}]")
 
    # By T.I.M.E. tags
    tag_counts = {"T": 0, "I": 0, "M": 0, "E": 0, "none": 0}
    for d in documents:
        tags = d.metadata.get("time_tags", "none")
        for tag in ["T", "I", "M", "E"]:
            if tag in tags.split(","):
                tag_counts[tag] += 1
        if tags == "none":
            tag_counts["none"] += 1
 
    print(f"\n🏷️  T.I.M.E. tag coverage:")
    print(f"   T (Tissue)    : {tag_counts['T']:>3} chunks")
    print(f"   I (Infection) : {tag_counts['I']:>3} chunks")
    print(f"   M (Moisture)  : {tag_counts['M']:>3} chunks")
    print(f"   E (Edge)      : {tag_counts['E']:>3} chunks")
    print(f"   No tags       : {tag_counts['none']:>3} chunks")
 
    # Processing type
    proc = Counter(d.metadata.get("processing", "unknown") for d in documents)
    print(f"\n⚙️  Processing breakdown:")
    for k, v in proc.items():
        print(f"   {v:>3}  {k}")
 
    # Content flags
    has_table = sum(1 for d in documents if d.metadata.get("has_table") == "True")
    has_image = sum(1 for d in documents if d.metadata.get("has_image") == "True")
    print(f"\n📊 Content flags:")
    print(f"   Has table : {has_table}")
    print(f"   Has image : {has_image}")
 
    # Char count stats
    char_counts = [len(d.page_content) for d in documents]
    print(f"\n📏 Chunk size stats:")
    print(f"   Min   : {min(char_counts):,} chars")
    print(f"   Max   : {max(char_counts):,} chars")
    print(f"   Mean  : {sum(char_counts)//len(char_counts):,} chars")
    print(f"   Total : {sum(char_counts):,} chars")
 
    print("\n" + "═" * 60)
 
 
print("✅ Diagnostics helper ready")

✅ Diagnostics helper ready


## 10. Main pipeline function

In [24]:
def run_curated_ingestion_pipeline(
    chunk_json_files: dict,
    persist_directory: str,
    guideline_metadata: dict,
    force_rebuild: bool = False,
) -> Chroma:
    """
    Full v2 ingestion pipeline:
 
      1. Load curated chunk JSONs (exported from chunk_visualiser)
      2. Convert to LangChain Documents:
           - Text-only chunks → use raw text directly
           - Table/image chunks → wound-care AI extraction via GPT-4o-mini
           - Non-clinical chunks → dropped (LLM flags SKIP)
      3. Re-extract T.I.M.E. tags on final content
      4. Attach per-source guideline provenance metadata
      5. Embed with MedEmbed-large-v0.1 and store in ChromaDB (cosine similarity)
 
    Parameters
    ----------
    chunk_json_files   : dict {label: path} of *_kept.json files
    persist_directory  : ChromaDB directory (will be created or overwritten)
    guideline_metadata : per-source metadata dict (authority, year, etc.)
    force_rebuild      : if True, deletes existing DB before building
 
    Returns
    -------
    Chroma vectorstore instance
    """
    print("🚀 VerdaSense RAG Ingestion Pipeline v2")
    print("=" * 60)
 
    # ── Guard: warn if DB exists ───────────────────────────────────────────────
    if os.path.isdir(persist_directory):
        if force_rebuild:
            import shutil
            print(f"⚠️  Deleting existing DB at {persist_directory} (force_rebuild=True)...")
            shutil.rmtree(persist_directory)
        else:
            print(f"⚠️  DB already exists at {persist_directory}")
            print(f"   Pass force_rebuild=True to delete and rebuild.")
            print(f"   Loading existing DB instead...")
            embedding_model = HuggingFaceEmbeddings(
                model_name="abhinand/MedEmbed-large-v0.1",
                model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"},
                encode_kwargs={"normalize_embeddings": True},
            )
            return Chroma(
                persist_directory=persist_directory,
                embedding_function=embedding_model,
                collection_metadata={"hnsw:space": "cosine"},
            )
 
    # ── STEP 1: Load curated chunks ────────────────────────────────────────────
    print("\n📂 STEP 1 — Loading curated chunk JSONs")
    print("-" * 40)
    curated_chunks = load_curated_chunks(list(chunk_json_files.values()))
 
    if not curated_chunks:
        print("❌ No chunks loaded. Check your JSON file paths.")
        return None
 
    # ── STEP 2: Convert to Documents ──────────────────────────────────────────
    print("\n🧠 STEP 2 — Converting chunks to LangChain Documents")
    print("-" * 40)
    documents = convert_curated_chunks_to_documents(
        curated_chunks   = curated_chunks,
        guideline_metadata = guideline_metadata,
    )
 
    if not documents:
        print("❌ No documents produced. Check chunk content.")
        return None
 
    # ── STEP 3: Diagnostics ────────────────────────────────────────────────────
    print("\n📊 STEP 3 — Pipeline diagnostics")
    print_pipeline_summary(documents)
 
    # ── STEP 4: Build vector store ─────────────────────────────────────────────
    print("\n🔮 STEP 4 — Building ChromaDB vector store")
    print("-" * 40)
    db = create_vector_store(documents, persist_directory=persist_directory)
 
    # ── STEP 5: Verify ────────────────────────────────────────────────────────
    print("\n✅ STEP 5 — Verification")
    print("-" * 40)
    collection = db.get()
    indexed_count = len(collection.get("documents", []))
    print(f"   Documents in ChromaDB : {indexed_count}")
    print(f"   Persist directory     : {persist_directory}")
 
    # Quick retrieval test
    test_query = "hydrogel dressing for dry necrotic wound"
    results = db.similarity_search(test_query, k=3)
    print(f"\n   Test query: '{test_query}'")
    print(f"   Top 3 results:")
    for i, r in enumerate(results, 1):
        src  = r.metadata.get("source", "?")
        tags = r.metadata.get("time_tags", "none")
        print(f"     {i}. [{tags}] {src}: {r.page_content[:80]}...")
 
    print(f"\n🎉 Ingestion pipeline v2 complete!")
    print(f"   Total indexed : {indexed_count} documents")
    print(f"   Vector store  : {persist_directory}")
    return db
 
 
print("✅ Main pipeline function ready")
print("\n▶  Run Cell 11 to execute the full pipeline.")

✅ Main pipeline function ready

▶  Run Cell 11 to execute the full pipeline.


## 11. Execute

In [25]:
db_v2 = run_curated_ingestion_pipeline(
    chunk_json_files   = CHUNK_JSON_FILES,
    persist_directory  = DB_DIR,
    guideline_metadata = GUIDELINE_METADATA,
    force_rebuild      = True,
)

🚀 VerdaSense RAG Ingestion Pipeline v2

📂 STEP 1 — Loading curated chunk JSONs
----------------------------------------
   📄 AJGP_kept.json: 12 kept chunks (of 12 total, 0 skipped)
   📄 GarisPanduan_kept.json: 5 kept chunks (of 29 total, 24 skipped)
   📄 SFP_kept.json: 128 kept chunks (of 128 total, 0 skipped)
   📄 WoundCare_kept.json: 78 kept chunks (of 146 total, 68 skipped)

   Total curated chunks loaded: 223

🧠 STEP 2 — Converting chunks to LangChain Documents
----------------------------------------
🧠 Converting 223 curated chunks to Documents...
   [  1/223] AJGP-11-2022-Focus-Sinha-Wound-Dressings — chunk #0 (img) 755 chars
     → AI extraction (has_table=False, has_image=True)...
     → Preview: - **Wound Characteristics and Dressing Recommendations:**
  - **Tissue Type:** Superficial open wounds (ulcers) require ...
     → AI extraction (has_table=False, has_image=True)...
     → Preview: - **Wound Characteristics and Dressing Recommendations:**
  - The article does not speci

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 4283.11it/s]


   Persisting to: ./db_wound_care_v2
   Indexing 202 documents...
✅ Vector store saved → ./db_wound_care_v2

✅ STEP 5 — Verification
----------------------------------------
   Documents in ChromaDB : 202
   Persist directory     : ./db_wound_care_v2

   Test query: 'hydrogel dressing for dry necrotic wound'
   Top 3 results:
     1. [T,M] SFP-Vol403.Unit2.pdf: brown and malodorous exudate often mistaken for infective

exudates10,11.

Hydro...
     2. [T,I,M] Wound_Care_Manual.pdf: - **Wound Characteristics and Dressing Recommendations:**
  - **Wet, infected wo...
     3. [T,M,E] SFP-Vol403.Unit2.pdf: clinical judgment. If the dressing is soiled, loose, slipping or

curling at the...

🎉 Ingestion pipeline v2 complete!
   Total indexed : 202 documents
   Vector store  : ./db_wound_care_v2
